# NSCLC_Toxicity: real CT to final prediction

This notebook downloads the **real public 3D Slicer CTChest scan**, creates five lung-lobe masks, measures all seven QCT features, computes DINOv3 outputs, and runs the final model. No synthetic imaging features are used.

The scan has no accompanying SBRT clinical information. The example tumor size, stage, location, histology, dose, and treatment timing are **fictional**, for demonstrating the software only. Results are not that person’s clinical risk.

Create and activate the environment in the README, then choose **Run → Run All Cells**. The first run downloads the CT and network weights; CPU segmentation can take several minutes. Internet is needed for downloads; image processing happens locally.


In [ ]:
from pathlib import Path
import os

root = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
if not (root / "nsclc_toxicity").is_dir():
    raise FileNotFoundError("Start Jupyter from the NSCLC_Toxicity repository folder.")
os.chdir(root)
# Limit CPU threads so the notebook remains usable on a laptop.
import torch
torch.set_num_threads(4)
print("Working folder:", root.name)


## 1. Download the public chest CT

The source is 3D Slicer’s official Sample Data. Its published SHA256 is verified. `source.json` records the download, attribution, and which values are fictional. A pre-existing clinical CSV is preserved.


In [ ]:
from nsclc_toxicity.public_ct import download_demo

manifest = download_demo(Path("data/public_ct"))


## 2. Review the clinical inputs

For your own case, provide these columns with real clinical values. Tumor location, histology, treatment dose, and timing cannot be reliably recovered from the CT alone. Leave unknown clinical values blank. The model converts T1c to stage 1 and histology/location to the required binary features.


In [ ]:
import pandas as pd

clinical = pd.read_csv(manifest)
clinical


## 3. Download the model weights

The three trained CT networks and final tabular/fusion model are downloaded with checksum verification. The segmentation model is fetched automatically at the next step.


In [ ]:
from nsclc_toxicity.download import download_weights

download_weights(Path("weights"), imaging=True)


## 4. Compute the imaging inputs from the raw scan

This converts NRRD to NIfTI without changing HU, generates the five lobe masks with the free TotalSegmentator `total` task, matches the model’s LAS orientation, measures Q7, resizes/window-scales the CT, and runs all three DINOv3 networks. Masks are saved on the original CT grid. Completed matching segmentations are reused on later runs.


In [ ]:
from nsclc_toxicity.imaging import prepare_ct

output = Path("results/public_ct")
prepare_ct(manifest, Path("weights"), output / "prepared.csv", device="cpu", auto_masks=True)


## 5. Inspect the measured features and network outputs

These numbers were calculated from the downloaded CT and its masks. No randomly generated values or prepared imaging tables enter this path.


In [ ]:
from nsclc_toxicity.schema import Q_FEATURES, FOLDS, dl_columns

prepared = pd.read_csv(output / "prepared.csv")
assert prepared[list(Q_FEATURES)].notna().all().all(), "Check lung masks and treated lobe."
prepared[["scan_id", *Q_FEATURES]].T


In [ ]:
hazard_columns = [column for fold in FOLDS for column in dl_columns(fold)]
assert prepared[hazard_columns].ge(0).all().all() and prepared[hazard_columns].le(1).all().all()
prepared[["scan_id", *hazard_columns]].T


## 6. Review a CT slice and the lobe masks

This preview helps inspect image coverage and mask placement; review the whole volume before using your own data. Each colored region is an automatically generated lung lobe.


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from nsclc_toxicity.imaging import LOBES, load_ct

ct_file = next((output / "converted_ct").glob("*.nii.gz"))
mask_dir = next((output / "lobe_masks").glob("*/lung_upper_lobe_right.nii.gz")).parent
ct = load_ct(ct_file).get_fdata(dtype=np.float32)
labels = np.zeros(ct.shape, dtype=np.uint8)
for number, name in enumerate(LOBES.values(), 1):
    mask = load_ct(mask_dir / f"{name}.nii.gz").get_fdata() > 0
    labels[mask] = number
z = int(np.argmax((labels > 0).sum(axis=(0, 1))))
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(ct[:, :, z].T, origin="lower", cmap="gray", vmin=-1000, vmax=400)
ax.imshow(np.ma.masked_equal(labels[:, :, z].T, 0), origin="lower", cmap="tab10", vmin=1, vmax=5, alpha=0.4, interpolation="nearest")
ax.set_title("Public CTChest: computed lung-lobe masks")
ax.axis("off")
fig.tight_layout()
fig.savefig(output / "mask_preview.png", dpi=150)
plt.show()


## 7. Run the final three-model ensemble

Scores use the measured QCT values, CT network outputs, and the example clinical inputs. They are uncalibrated research scores, not risk percentages. Only the `Multimodal` rows are displayed below; the saved CSV also contains the component branches.


In [ ]:
from nsclc_toxicity.model import load_model, predict

predictions = predict(prepared, load_model("weights/final_model.json"))
predictions.to_csv(output / "predictions.csv", index=False)
final = predictions.loc[predictions["model"].eq("Multimodal")]
assert len(final) == 4
final[["scan_id", "horizon_days", "score"]]


## Your own CT

Copy `data/clinical_manifest_example.csv` into `private_data/clinical.csv`. Set `ct_path` to your HU NIfTI/NRRD file, or to a folder containing exactly one DICOM CT series. Paths are relative to the clinical CSV. Enter the clinical values and treatment-relative scan day. Then run:

```bash
python -m nsclc_toxicity infer-ct --manifest private_data/clinical.csv --output results/my_case
```

No masks or imaging scores need to be supplied. See `docs/CT_WORKFLOW.md` for follow-up scans, full-resolution segmentation, and externally reviewed masks. The separate `csv_demo.ipynb` demonstrates tabular experiments on synthetic data.
